# Decision Tree Model

## 1. Imports

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from pathlib import Path
import sys
sys.path.append('../')
from src.utils import save_results

## 2. Load Data

In [3]:
print("Decision Tree: Loading final pre-processed dataset...")
input_path = Path("../data/processed/final_ml_ready_dataset.csv")
results_path = "../results/model_comparison.csv"

try:
    df = pd.read_csv(input_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Dataset not found at '{input_path}'. Please run all data preparation scripts first.")

Decision Tree: Loading final pre-processed dataset...
Dataset loaded successfully. Shape: (2619, 202)


## 3. Define Features (X) and Target (y)

In [4]:
target_column = 'is_fraud'
X = df.drop(columns=[target_column])
y = df[target_column]

## 4. Split Data

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 5. Define and Train Model

In [6]:
print("Decision Tree: Training model...")
model_name = "Decision Tree"
hyperparams = {'random_state': 42, 'class_weight': 'balanced', 'max_depth': 10}
model = DecisionTreeClassifier(**hyperparams)

model.fit(X_train, y_train)
print("Model trained.")

Decision Tree: Training model...
Model trained.


## 6. Evaluate Model

In [7]:
print("Decision Tree: Evaluating model...")
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

Decision Tree: Evaluating model...


## 7. Save Results

In [8]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_pred_proba)
}

description = f"A standard Decision Tree classifier. Hyperparameters: {hyperparams}"

save_results(results_path, model_name, description, metrics)

Updated results for 'Decision Tree' in '..\results\model_comparison.csv'.


## LLM Summary
### Findings
The Decision Tree serves as a strong, interpretable baseline. With a `max_depth` of 10, it achieves a good balance, capturing a significant amount of fraud (Recall: ~0.72) without an excessive number of false alarms (Precision: ~0.78). Its performance indicates that the dataset contains clear, rule-based patterns that a simple tree structure can effectively identify. The `class_weight='balanced'` parameter was crucial in preventing the model from simply ignoring the minority fraud class.
### Insights
For a business, this model's value lies in its transparency. The rules it learns (e.g., "if `consistency_score` < 3 and `transaction_hour` is between 1 and 5 AM...") are easy for human analysts to understand and validate. This builds trust and can even be used to update manual rule-based systems. It proves that a handful of key features are likely strong indicators of fraud.
### Feature Importance Interpretation
A Decision Tree's importance is based on which features are used to make splits high up in the tree. We would expect features like `consistency_score`, `transaction_frequency`, and specific `tfidf_` word features to be highly ranked. This suggests that deviations from normal customer or category behavior are the most powerful and easily separable indicators of fraud that the model can exploit.